In [ ]:
# Cell 1 — Install
!pip install -q transformers accelerate flask pyngrok torch sentencepiece

In [ ]:
# Cell 2 — HuggingFace auth (need access to meta-llama/Meta-Llama-3-8B)
from huggingface_hub import login
login("Your_huggingface_token")   # or use Kaggle secrets

# Cell 3 — ngrok token for Instance 2 (must be a separate ngrok account)
import os
os.environ["NGROK_TOKEN"] = "Your_ngrok_token1" #microsoft-edge-avinash

In [ ]:
"""
╔══════════════════════════════════════════════════════════╗
║         LLAMA 3 8B — PIPELINE STAGE 0                   ║
║         Kaggle Instance 0                                ║
║         Layers: Embed + 0–10                             ║
║         Role: Tokenize → Forward → Send hidden states   ║
╚══════════════════════════════════════════════════════════╝

SETUP (run these cells first in Kaggle):
  !pip install -q transformers accelerate flask pyngrok torch sentencepiece
  !huggingface-cli login  # paste your HF token when prompted
"""

# ── Imports ──────────────────────────────────────────────────────────────────
import os, io, time, threading, requests
import torch
import numpy as np
from flask import Flask, request, jsonify, Response, stream_with_context
from transformers import AutoTokenizer, AutoModelForCausalLM
from pyngrok import ngrok

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID       = "meta-llama/Meta-Llama-3-8B-Instruct"   # or "meta-llama/Meta-Llama-3-8B-Instruct"
SPLIT_LAYER    = 11                              # Stage 0 owns layers 0..10 (inclusive)
STAGE1_URL     = None                            # Set after Stage 1 boots (see below)
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE          = torch.float16
MAX_NEW_TOKENS = 200
TEMPERATURE    = 0.8
TOP_P          = 0.95

print(f"[Stage 0] Device: {DEVICE}  |  dtype: {DTYPE}")

# ── Model Loading ─────────────────────────────────────────────────────────────
def load_stage0_model():
    print("[Stage 0] Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    print(f"[Stage 0] Loading full model then slicing to layers 0–{SPLIT_LAYER - 1}...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=DTYPE,
        device_map="auto",
    )

    # Slice to only our layers
    model.model.layers = model.model.layers[:SPLIT_LAYER]

    # Replace with identity instead of deleting
    model.model.norm = torch.nn.Identity()
    model.lm_head = torch.nn.Identity()

    # ✅ After slicing, device_map="auto" hooks are stale — move everything to GPU cleanly
    model = model.to(DEVICE)

    model.eval()
    print(f"[Stage 0] Model loaded. VRAM used: "
          f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")
    return model, tokenizer


# ── KV-Cache Store ────────────────────────────────────────────────────────────
# We keep a simple in-memory dict: session_id -> {"pkv_0": <past_key_values>}
# For research/single-session use this is fine.
SESSIONS = {}

def tensor_to_bytes(t: torch.Tensor) -> bytes:
    """Serialize tensor to raw bytes (numpy framing)."""
    arr = t.cpu().to(torch.float16).numpy()
    buf = io.BytesIO()
    np.save(buf, arr)
    return buf.getvalue()

def bytes_to_tensor(b: bytes) -> torch.Tensor:
    buf = io.BytesIO(b)
    arr = np.load(buf)
    return torch.from_numpy(arr).to(DTYPE).to(DEVICE)

# ── Flask App ─────────────────────────────────────────────────────────────────
app = Flask(__name__)

@app.route("/health")
def health():
    return jsonify({"stage": 0, "status": "ok",
                    "vram_gb": torch.cuda.memory_allocated() / 1e9})

@app.route("/generate", methods=["POST"])
def generate():
    data = request.json
    prompt         = data["prompt"]
    max_new_tokens = data.get("max_new_tokens", MAX_NEW_TOKENS)
    session_id     = data.get("session_id", "default")
    temperature    = data.get("temperature", TEMPERATURE)
    top_p          = data.get("top_p", TOP_P)

    if STAGE1_URL is None:
        return jsonify({"error": "STAGE1_URL not set."}), 503

    def generate_stream():
        global SESSIONS
        tokenizer_local = tokenizer
        model_local     = model

        # Tokenize
        input_ids = tokenizer_local(prompt, return_tensors="pt").input_ids.to(DEVICE)
        generated_ids = []

        # Retrieve or init KV cache
        pkv_0 = SESSIONS.get(session_id, {}).get("pkv_0", None)

        t_start = time.time()
        for step in range(max_new_tokens):
            # ── Stage 0 forward ──────────────────────────────────────────────
            # In generate_stream():
            with torch.no_grad():
                if step == 0:
                    ids_in = input_ids
                else:
                    ids_in = torch.tensor([[next_token]], device=DEVICE)
            
                out    = model_local(ids_in, past_key_values=pkv_0,
                                     use_cache=True, output_hidden_states=True)
                pkv_0  = out.past_key_values
                hidden = out.hidden_states[-1]   # (1, seq_len, hidden_size) on step 0
                                                 # (1, 1, hidden_size) on step 1+
            
            current_position = pkv_0.get_seq_length() - 1
            
            # ✅ Send FULL hidden states on step 0 (all prompt positions)
            # so downstream stages can fill their KV caches
            # On step 1+ send only last position as before
            payload_bytes = tensor_to_bytes(hidden)   # full sequence on step 0
            
            seq_len_sent = hidden.shape[1]
            
            resp = requests.post(
                f"{STAGE1_URL}/forward",
                data=payload_bytes,
                headers={
                    "Content-Type":  "application/octet-stream",
                    "X-Session-Id":  session_id,
                    "X-Step":        str(step),
                    "X-Temperature": str(temperature),
                    "X-Top-P":       str(top_p),
                    "X-Past-Len":    str(current_position - seq_len_sent + 1),  # start position
                    "X-Seq-Len":     str(seq_len_sent),
                },
                timeout=60,
            )
            resp.raise_for_status()
            result     = resp.json()
            next_token = result["token_id"]

            # ── Decode and stream ─────────────────────────────────────────────
            tok_str = tokenizer_local.decode([next_token], skip_special_tokens=True)
            generated_ids.append(next_token)
            yield tok_str

            # ── EOS check ────────────────────────────────────────────────────
            if next_token == tokenizer_local.eos_token_id:
                break

        # Save session KV cache
        SESSIONS[session_id] = {"pkv_0": pkv_0}

        elapsed = time.time() - t_start
        tps     = len(generated_ids) / elapsed
        print(f"[Stage 0] Generated {len(generated_ids)} tokens in "
              f"{elapsed:.1f}s ({tps:.1f} tok/s)")

    return Response(stream_with_context(generate_stream()),
                    mimetype="text/plain")


# ── Boot Sequence ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # 1. Load model
    model, tokenizer = load_stage0_model()

    # 2. Set STAGE1_URL  ← YOU MUST FILL THIS IN after Stage 1 is running
    #    After running stage1_server.py in the other Kaggle notebook,
    #    copy the ngrok URL it prints and paste it here.
    STAGE1_URL = "https://a64e-136-113-115-96.ngrok-free.app"
    print(f"[Stage 0] Will forward hidden states to Stage 1 at: {STAGE1_URL}")

    # 3. Start ngrok tunnel for this server (so client can reach Stage 0)
    ngrok_token = os.environ.get("NGROK_TOKEN", "")   # set via Kaggle secrets
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
    tunnel = ngrok.connect(5000)
    print(f"\n{'='*60}")
    print(f"[Stage 0] PUBLIC URL: {tunnel.public_url}")
    print(f"  → Share this with your client (client.py)")
    print(f"  → POST to {tunnel.public_url}/generate")
    print(f"{'='*60}\n")

    # 4. Run Flask
    app.run(host="0.0.0.0", port=5000, threaded=False)